# Stage B validation — summary

Bundles the two endpoints of the validation story:
- **§8.2.1** internal-consistency *gate* — run **before** AERONET-blind tests to reject overfit/underfit candidates.
- **§8.2.6** success-criteria *scorecard* — the headline baseline / target / achieved table.

Prerequisites:
- Trained RF bundle at `MODELS_DIR/rf_primary.joblib`.
- B3 RF gap-filled per-slot NetCDFs in `RF_OUTPUT_DIR`.

If missing, run `python run_stage_b.py all --start 2022-09-01 --end 2026-04-30` first.

In [1]:
from datetime import date
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import validate as vb
import rf_gapfill as rf
import config as cfg

START = cfg.TEST_START
END   = cfg.TEST_END
print(f'Held-out window: {START} → {END}')

Held-out window: 2025-01-01 → 2026-04-30


## §8.2.1 — Internal consistency gate

Reads the train / CV / internal-test RMSEs stored on the trained RF bundle.  Checks are **one-sided** — train and test only fail when *worse* than CV by the tolerance margin.  Train *better* than CV is expected (the final fit uses every training day with no held-out block); a calmer chronological test slice may also legitimately beat the worst CV fold.

Pass criteria:
- `train_underfit_pass` — `rmse_train ≤ rmse_cv_mean · (1+tol)`.  Fails only if the model can't fit training data (true underfit signal).
- `test_overfit_pass` — `rmse_internal_test ≤ rmse_cv_mean · 1.3`.  Fails only when the held-out test is catastrophically worse than CV.

Also reports `rmse_cv_std`, `rmse_cv_max`, `worst_fold_idx` so a high `rmse_cv_mean` driven by one seasonal-hot-spot fold is visible.  Inspect `bundle.cv_fold_metrics[worst_fold_idx]` to see which time block was hardest.

In [2]:
bundle = rf.load_bundle("rf_residual_stctx_tau6")
print('Training window:', bundle.training_window)
print('Hyperparameters:', bundle.hyperparams)
print('Metrics       :', bundle.metrics)

diag = vb.internal_consistency(bundle.metrics, bundle.cv_fold_metrics)
pd.DataFrame([diag])

Training window: ('2022-09-01', '2024-12-31')
Hyperparameters: {'n_estimators': 200, 'max_depth': 20, 'min_samples_leaf': 5, 'max_features': 0.5}
Metrics       : {'n_train': 1366296, 'n_test': 337518, 'n_features': 28, 'rmse_train': 0.05843741341941351, 'rmse_cv_mean': 0.08224763961200829, 'rmse_cv_std': 0.015823476073714555, 'r2_cv_mean': 0.836322667273226, 'r2_cv_std': 0.043298838043862864, 'rmse_internal_test': 0.08940734726465233, 'r2_internal_test': 0.8093246855805628, 'rmse_train_aod': 0.05843741342293037, 'rmse_cv_mean_aod': 0.08224763961623197, 'rmse_cv_std_aod': 0.015823476035749323, 'r2_cv_mean_aod': 0.8875380049438819, 'r2_cv_std_aod': 0.043385039945627216, 'rmse_internal_test_aod': 0.08940734734722337, 'r2_internal_test_aod': 0.8621662047827789}


,rmse_train,rmse_cv_mean,rmse_cv_std,rmse_internal_test,tolerance,train_underfit_pass,test_overfit_pass,rmse_cv_max,worst_fold_idx
0,0.058437,0.082248,0.015823,0.089407,0.15,True,True,0.099372,2


## §8.2.6 — Success-criteria scorecard

Headline §9 table.  Achieved values come from the **full** RF product (`blind_only=False`) and the post-fill coverage audit.

In [3]:
pairs_full = vb.aeronet_pairs(START, END, candidate='rf', blind_only=False)
cov        = vb.coverage_audit(START, END)
vb.success_table(pairs_full, cov)

,metric,baseline,target,achieved
0,AERONET R Nghia Do (all matched slots),0.915,≥0.90,0.720276
1,AERONET R Bac Lieu (all matched slots),0.845,≥0.85,-0.067925
2,AERONET RMSE Nghia Do,0.271,≤0.30,0.536229
3,30-min AOD spatial coverage over Vietnam,0.103,≥95%,1.000000


In [4]:
pairs_full_kg = vb.aeronet_pairs(START, END, candidate='kriging', blind_only=False)
cov_kg        = vb.coverage_audit(START, END, candidate='kriging')
vb.success_table(pairs_full_kg, cov_kg)

,metric,baseline,target,achieved
0,AERONET R Nghia Do (all matched slots),0.915,≥0.90,0.845274
1,AERONET R Bac Lieu (all matched slots),0.845,≥0.85,0.208591
2,AERONET RMSE Nghia Do,0.271,≤0.30,0.402516
3,30-min AOD spatial coverage over Vietnam,0.103,≥95%,0.851533
